# Aula 01 — Recuperação de Informação: do problema da busca ao nosso motor
**Projeto Integrador III — Motor de Busca**
*Kernel: R | Módulos 1 a 10 (Teoria Completa e Teste Final)*

## Setup Inicial do Motor da Disciplina
Carrega as funções canônicas acumuladas (`motor01.R`) e inicializa o ambiente.

In [1]:
# Primeira celula do curso para Aula 01
source("https://raw.githubusercontent.com/fractalarea/pi3-motor-de-busca/main/motor/motor01.R")
estado()   # exibe MOTOR_VERSAO motor01 e a funcao tokenizar

[1] "MOTOR_VERSAO: motor01 (Aula 00 -> Aula 01)"
Funcoes carregadas: tokenizar, estado, anexar_estado.


## Módulos 1 e 2 — Recuperação de Informação e Arquitetura *Retrieve & Rerank*
- **RI:** Relevância em vez de igualdade, lista ordenada em vez de conjunto, consulta como tradução aproximada da necessidade.
- **Arquitetura:** Estágio 1 rápido e barato (filtra milhões para centenas) + Estágio 2 preciso (rerank neural).

## Módulo 3 — O corpus canônico em R
O vetor nomeado `docs` com as 8 frases canônicas da disciplina.

In [2]:
docs <- c(
  d1 = "recuperacao de informacao ordena documentos por relevancia",
  d2 = "o modelo de espaco vetorial representa documentos como vetores",
  d3 = "bm25 e um modelo probabilistico de ranqueamento de texto",
  d4 = "aprendizado estatistico fundamenta a recuperacao moderna",
  d5 = "o indice invertido acelera a busca em muitos documentos",
  d6 = "embeddings capturam a semantica de palavras e documentos",
  d7 = "a avaliacao mede a relevancia dos resultados da busca",
  d8 = "ciencia de dados combina estatistica e programacao"
)

length(docs)
docs["d5"]

# Checkpoint 3
nchar(docs[["d8"]])
nchar(docs["d8"])

[1] 8
                                                       d5 
"o indice invertido acelera a busca em muitos documentos" 
[1] 50
d8 
50 


## Módulo 4 — Tokenização
Aplicação da função `tokenizar` sobre o corpus através de `lapply`.

In [3]:
tokens <- lapply(docs, tokenizar)
tokens[["d1"]]
sapply(tokens, length)

# Checkpoint 4
t_teste <- tokenizar("Modelo modelo MODELO")
t_teste
length(t_teste)
length(unique(t_teste))

[1] "recuperacao" "de"          "informacao"  "ordena"      "documentos" 
[6] "por"         "relevancia" 
d1 d2 d3 d4 d5 d6 d7 d8 
 7  9  9  6  9  8  9  7 
[1] "modelo" "modelo" "modelo"
[1] 3
[1] 1


## Módulo 5 — Vocabulário e Frequência Global
Extração dos termos únicos do corpus (`vocab`) e contagem com `table`.

In [4]:
vocab <- sort(unique(unlist(tokens)))
length(vocab)   # 45 termos distintos

freq <- table(unlist(tokens))
sort(freq, decreasing = TRUE)[1:6]

# Checkpoint 5: cópia com d9='busca busca busca'
docs9_teste <- c(docs, d9 = "busca busca busca")
tokens9 <- lapply(docs9_teste, tokenizar)
table(unlist(tokens9))["busca"] # vai de 2 para 5
length(unique(unlist(tokens9))) # vocabulário permanece 45

[1] 45

        de          a documentos          e      busca     modelo 
         6          5          4          3          2          2 
busca 
    5 
[1] 45


## Módulo 6 — Matriz Termo-Documento (TDM)
Construção da TDM $45 \times 8$ com `factor(levels = vocab)` para uniformizar o tamanho das colunas.

In [5]:
tdm <- sapply(tokens, function(tk) {
  as.integer(table(factor(tk, levels = vocab)))
})
rownames(tdm) <- vocab

# Visualizar primeiras linhas
tdm[1:6, ]
dim(tdm)
colSums(tdm)

            d1 d2 d3 d4 d5 d6 d7 d8
a            0  0  0  1  1  1  2  0
acelera      0  0  0  0  1  0  0  0
aprendizado  0  0  0  1  0  0  0  0
avaliacao    0  0  0  0  0  0  1  0
bm25         0  0  1  0  0  0  0  0
busca        0  0  0  0  1  0  1  0
[1] 45  8
d1 d2 d3 d4 d5 d6 d7 d8 
 7  9  9  6  9  8  9  7 


## Módulo 7 — Busca Booleana e Limitação da Falta de Ordenação
Implementação da busca booleana exata e composição conjuntiva com `intersect`.

In [6]:
busca_booleana <- function(termo, tdm) {
  termo <- tolower(termo)
  if (!termo %in% rownames(tdm)) return(character(0))
  colnames(tdm)[tdm[termo, ] > 0]
}

busca_booleana("modelo", tdm)
busca_booleana("de", tdm)
busca_booleana("XYZ", tdm)

# Checkpoint 7: 'documentos E busca'
intersect(busca_booleana("documentos", tdm), busca_booleana("busca", tdm))

[1] "d2" "d3"
[1] "d1" "d2" "d3" "d6" "d8"
character(0)
[1] "d5"


## Módulos 8 e 9 — Ponderação TF-IDF e Reciclagem Matricial
Cálculo dos pesos locais (TF) e globais (IDF = $\log(N/\text{df})$), multiplicados por reciclagem.

In [7]:
N   <- ncol(tdm)             # 8 documentos
df  <- rowSums(tdm > 0)      # contagem de documentos em que cada termo aparece
idf <- log(N / df)           # vetor de 45 pesos idf
tfidf <- tdm * idf           # multiplicacao por reciclagem matricial

# Inspecionar idf de termos selecionados
round(idf[c("de", "documentos", "recuperacao", "bm25")], 2)

# Visualizar pesos de 4 termos nos 8 documentos
round(tfidf[c("documentos", "recuperacao", "busca", "de"), ], 2)

# Checkpoint 9: tfidf de 'modelo' em d3
cat("Peso tfidf de modelo em d3:\n")
tfidf["modelo", "d3"]

         de documentos recuperacao        bm25 
       0.47        0.69        1.39        2.08 
              d1   d2   d3   d4   d5   d6   d7   d8
documentos  0.69 0.69 0.00 0.00 0.69 0.69 0.00 0.00
recuperacao 1.39 0.00 0.00 1.39 0.00 0.00 0.00 0.00
busca       0.00 0.00 0.00 0.00 1.39 0.00 1.39 0.00
de          0.47 0.47 0.94 0.00 0.00 0.47 0.00 0.47
Peso tfidf de modelo em d3:
[1] 1.386294


## Módulo 10 — Regex vs Motor de Busca
- Regex: busca padrões sequenciais em caracteres (pedaço).
- Motor: busca e pondera unidades conceituais estruturadas (termos).

In [8]:
grep("recupera", docs)          # acha por subcadeia
busca_booleana("recupera", tdm) # nao acha pois o termo e 'recuperacao'

# Checkpoint 10
grep("busca", docs)
busca_booleana("busc", tdm)

[1] 1 4
character(0)
[1] 5 7
character(0)


## Fechamento da Sessão e Snapshot de Estado
Anexa a memória da sessão da Aula 01 ao consolidado.

In [9]:
if ("aula01_consolidado.md" %in% list.files()) {
  anexar_estado("aula01_consolidado.md")
  cat("Estado anexado com sucesso a aula01_consolidado.md.\n")
} else {
  cat("Snapshot de estado concluido para aula01_consolidado.md.\n")
}

Snapshot de estado concluido para aula01_consolidado.md.
